# LangGraph MLOps Intelligence Agent — Training Notebook

This notebook is the executable design lab for an agentic MLOps control plane.

It combines Pandas, NumPy, Matplotlib, LangGraph, MLflow/Prometheus/Kubernetes integration points, and human approval patterns.


In [ ]:
# Uncomment if needed:
# %pip install -q langgraph langchain langchain-openai pydantic mlflow prometheus-api-client kubernetes


In [ ]:
from typing import TypedDict, Literal, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Synthetic MLOps telemetry


In [ ]:
telemetry = pd.DataFrame([
    {'timestamp':'14:00','gpu_util':71,'queue_depth':3,'latency_ms':128,'f1':0.914,'data_drift':0.08},
    {'timestamp':'14:05','gpu_util':76,'queue_depth':4,'latency_ms':135,'f1':0.912,'data_drift':0.10},
    {'timestamp':'14:10','gpu_util':84,'queue_depth':7,'latency_ms':149,'f1':0.908,'data_drift':0.14},
    {'timestamp':'14:15','gpu_util':91,'queue_depth':11,'latency_ms':181,'f1':0.901,'data_drift':0.19},
    {'timestamp':'14:20','gpu_util':97,'queue_depth':17,'latency_ms':233,'f1':0.892,'data_drift':0.27},
])
telemetry


In [ ]:
summary = {
    'gpu_mean': float(np.mean(telemetry['gpu_util'])),
    'gpu_max': int(np.max(telemetry['gpu_util'])),
    'queue_latest': int(telemetry['queue_depth'].iloc[-1]),
    'latency_latest': int(telemetry['latency_ms'].iloc[-1]),
    'f1_change': float(telemetry['f1'].iloc[-1] - telemetry['f1'].iloc[0]),
    'drift_latest': float(telemetry['data_drift'].iloc[-1]),
}
summary


In [ ]:
fig, ax = plt.subplots()
ax.plot(telemetry['timestamp'], telemetry['queue_depth'], marker='o')
ax.set_title('Training queue depth')
ax.set_xlabel('Time')
ax.set_ylabel('Queued jobs')
plt.show()


## LangGraph state and diagnostic nodes


In [ ]:
class MLOpsState(TypedDict, total=False):
    telemetry_summary: dict
    issue_type: Literal['healthy','performance','scaling','model','data']
    severity: Literal['info','warning','critical']
    diagnosis: str
    recommendation: str
    requires_human: bool
    approved: Optional[bool]

def inspect_pipeline(state: MLOpsState):
    return {'telemetry_summary': summary}

def classify_issue(state: MLOpsState):
    s = state['telemetry_summary']
    if s['drift_latest'] >= 0.25 and s['f1_change'] < -0.015:
        return {'issue_type':'model','severity':'critical'}
    if s['gpu_max'] >= 95 and s['queue_latest'] >= 10:
        return {'issue_type':'scaling','severity':'warning'}
    if s['latency_latest'] >= 200:
        return {'issue_type':'performance','severity':'warning'}
    return {'issue_type':'healthy','severity':'info'}

def diagnose_model(state: MLOpsState):
    return {'diagnosis':'Model quality is declining while data drift is increasing.',
            'recommendation':'Run challenger evaluation and inspect feature drift before promotion.',
            'requires_human':True}

def diagnose_scaling(state: MLOpsState):
    return {'diagnosis':'GPU saturation and queue growth indicate constrained capacity.',
            'recommendation':'Scale workers within policy limits and review queued experiments.',
            'requires_human':False}

def diagnose_performance(state: MLOpsState):
    return {'diagnosis':'Latency exceeded the warning threshold.',
            'recommendation':'Inspect resource saturation, queueing, and recent deployments.',
            'requires_human':False}

def healthy(state: MLOpsState):
    return {'diagnosis':'No material anomaly detected.', 'recommendation':'Continue monitoring.', 'requires_human':False}

def route_issue(state: MLOpsState):
    return state['issue_type']


## Build and run the LangGraph


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(MLOpsState)
builder.add_node('inspect', inspect_pipeline)
builder.add_node('classify', classify_issue)
builder.add_node('model', diagnose_model)
builder.add_node('scaling', diagnose_scaling)
builder.add_node('performance', diagnose_performance)
builder.add_node('healthy', healthy)
builder.add_node('data', diagnose_model)  # placeholder
builder.add_edge(START, 'inspect')
builder.add_edge('inspect', 'classify')
builder.add_conditional_edges('classify', route_issue, {
    'model':'model','scaling':'scaling','performance':'performance','data':'data','healthy':'healthy'})
for node in ['model','scaling','performance','data','healthy']:
    builder.add_edge(node, END)
graph = builder.compile()
graph.invoke({})


## Controlled ML/ops tool stubs

These become real MLflow, Prometheus, Kubernetes, scikit-learn, PyTorch, and TensorFlow calls later.


In [ ]:
def query_mlflow_metrics(model_name: str):
    return {'model':model_name,'f1':0.892,'status':'demo'}

def query_prometheus(metric: str):
    return {'metric':metric,'value':97,'status':'demo'}

def inspect_kubernetes(workload: str):
    return {'workload':workload,'replicas':2,'status':'demo'}

def launch_challenger_evaluation(model_name: str):
    return {'model':model_name,'action':'evaluation_requested','status':'demo'}


## Notebook → production promotion path

Once a section is stable:

1. Move state models into `src/agent/state.py`.
2. Move nodes into `src/agent/nodes/`.
3. Move tool wrappers into `src/tools/`.
4. Add persistent checkpointing.
5. Expose the graph through FastAPI.
6. Put heavy ML execution into separate worker containers.
7. Deploy with Helm/Kubernetes.
8. Connect live MLflow, Prometheus, and Kubernetes clients.
9. Add human approval for sensitive actions.
